# 09 — Capstone: policy-derived ground truth

**Estimated time:** 50 minutes<br>
**Prerequisites:** 08 — Local MLflow evidence and promotion decisions<br>
**Learner-produced evidence:** a versioned 400/100/150 policy dataset and deterministic ceiling

## Learning objectives

- Separate deterministic, policy, external-lookup, and human-judgment rules.
- Generate controlled ground truth without asking an LLM to invent critical labels.
- Verify capstone split counts, slice coverage, hashes, and the deterministic ceiling.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

The capstone models a policy-driven document review. Here the correct answer depends on an explicit policy and known synthetic facts, so ground truth can be generated deterministically. That is different from asking an LLM to invent labels or assuming a convenient public dataset represents the domain. The quality ceiling is set by the authority of the labels, not by the number of rows.

## Key terms in plain language

- **ground truth:** the authoritative expected value used for evaluation, with a documented source and limitations.
- **policy engine:** deterministic code that applies versioned policy rules to validated facts.
- **external lookup:** a governed source outside the document, such as an approved vendor or jurisdiction registry.
- **human judgment:** a decision supplied by a qualified reviewer when rules or available facts do not determine the answer.
- **synthetic data:** artificially generated examples whose construction process and assumptions are known.
- **controlled violation:** a deliberately introduced, labeled rule breach used to test coverage.
- **provenance per label:** metadata stating which rule, fact source, generator version, and seed produced an expected answer.
- **accuracy ceiling:** the maximum defensible performance given the available inputs and authority boundaries.


## Mental model — how to think about this

Draw an authority map before drawing a model architecture. Some facts are present in the document and can be validated. Some come only from an external registry. Some decisions are deterministic policy. Others require human judgment. Generate or label only what the chosen authority can actually know; mark everything else as unavailable or requiring review.

### Running example

For a deployment-readiness manifest, `owner_group` is either present or absent in validated input, and a versioned rule determines the corresponding check. Whether an external deployment really exists requires an authorized registry lookup. Whether its remaining business risk is acceptable belongs to a human—not to generated ground truth.

### Questions to ask before continuing

- For each target field, what is the authoritative source of truth?
- Can the answer be derived from provided facts and versioned policy without subjective interpretation?
- Which interactions and rare policy violations must the generated splits cover?
- What uncertainty or missing external fact should route to a human instead of receiving an invented label?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Generate critical labels from deterministic rules.** The LLM may help with non-authoritative language, but it must not create the policy truth used to grade itself.
- **Version the generator, policy, schema, seed, and rule catalog.** Reproducibility requires more than saving the final JSONL files.
- **Design coverage intentionally.** Include compliant cases, each rule violation, boundary values, missing facts, and interactions; publish counts by rule and split.
- **Hold out combinations, not merely rows.** Keep related templates and selected rule interactions out of training so evaluation tests composition rather than memorized wording.
- **Retain a human escalation class.** Unknown or subjective situations should remain explicit instead of being forced into confident binary labels.

## Common mistakes and why they fail

- **Asking an LLM to generate both examples and authoritative labels.** Correlated mistakes can create a convincing but circular benchmark.
- **Treating a nearby public dataset as domain ground truth.** Similar columns do not establish the same policy, jurisdiction, or intended use.
- **Mistaking synthetic volume for validity.** Thousands of rows from one narrow generator can repeat the same assumptions.
- **Skipping rule-coverage review.** Deterministic generation can still encode an incomplete or incorrect policy model.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Risk guidance:** [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)
- **Tool guidance:** [Hugging Face Dataset Cards documentation](https://huggingface.co/docs/hub/en/datasets-cards)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Why Kaggle is not the capstone ground truth

Customer-support records do not represent Databricks application
readiness. The capstone uses a small reviewed domain dataset generated
from an explicit policy engine. Every expected check carries rule kind,
source fields, facts origin, severity, and remediation provenance.


In [ ]:
from collections import Counter

import pandas as pd

from aai_local_finetuning.capstone import (
    REQUIRED_FROZEN_TEST_SLICES,
    evaluate_manifest,
    generate_capstone_dataset,
    load_capstone_records,
    render_capstone_mlx_dataset,
    rule_catalog,
)
from aai_local_finetuning.settings import PROJECT_ROOT

rules = rule_catalog()
pd.DataFrame(
    [
        {
            "rule": rule.rule_id,
            "kind": rule.kind.value,
            "severity": rule.failure_severity.value,
            "source_fields": ", ".join(rule.source_fields),
        }
        for rule in rules
    ]
)

## Rule kinds define authority

Deterministic rules read manifest facts. Policy rules apply a versioned
threshold or vocabulary. External lookups require an authorized system;
human judgment requires a person. The latter two route to review rather
than letting a tiny model invent facts.


In [ ]:
Counter(rule.kind.value for rule in rules)

## Generate reviewed combinations

A fixed seed creates controlled rule violations and interacting failures.
The frozen test includes required slices and unseen combinations of known
failures. Generation writes portable records and immutable hashes.


In [ ]:
source_dir = PROJECT_ROOT / "data" / "processed" / "capstone-readiness-v1"
mlx_dir = PROJECT_ROOT / "data" / "processed" / "capstone-mlx-v1"
split_manifest = generate_capstone_dataset(source_dir)
training_manifest = render_capstone_mlx_dataset(source_dir, mlx_dir)
{
    "counts": {
        artifact.split.value: artifact.record_count
        for artifact in split_manifest.artifacts
    },
    "frozen_test": split_manifest.frozen_test,
    "dataset_sha256": split_manifest.dataset_sha256,
    "portable_training_fingerprint": training_manifest.dataset_fingerprint,
}

## Inspect one synthetic example and its provenance

This is generated application metadata, not customer data. The expected
review comes entirely from the policy engine, including review routing
for facts that are unavailable locally.


In [ ]:
train_records = load_capstone_records(source_dir / "train.jsonl")
example = train_records[0]
{
    "example_id": example.example_id,
    "slices": example.metadata.slices,
    "manifest": example.manifest,
    "expected_status": example.expected_output.status.value,
    "non_pass_checks": [
        {
            "name": check.name,
            "result": check.result.value,
            "severity": check.severity.value,
            "rule_kind": check.provenance.rule_kind.value,
            "facts_origin": check.provenance.facts_origin,
        }
        for check in example.expected_output.checks
        if check.result.value != "pass"
    ],
}

## Lock the frozen contract without opening test rows

The generator commits the test count, file hash, ID hash, and required
slice vocabulary. We can inspect that contract without loading examples,
labels, or predictions. The deterministic engine is the *declared*
correctness ceiling for rules it fully determines; notebook 10 measures
that claim only after its model probe and optional training are finished.


In [ ]:
test_artifact = next(
    artifact for artifact in split_manifest.artifacts if artifact.split.value == "test"
)
{
    "test_rows_loaded": False,
    "frozen": split_manifest.frozen_test,
    "record_count": test_artifact.record_count,
    "sha256": test_artifact.sha256,
    "example_ids_sha256": test_artifact.example_ids_sha256,
    "required_slice_contract": list(REQUIRED_FROZEN_TEST_SLICES),
}

## Exercise — review a new manifest

Change one field and inspect which check changes. Success means you can
explain whether the outcome came from a manifest fact, platform policy,
an external system, or human review.


In [ ]:
learner_manifest = dict(example.manifest)
learner_manifest["owner"] = None
learner_review = evaluate_manifest(learner_manifest)
[
    {
        "name": check.name,
        "result": check.result.value,
        "kind": check.provenance.rule_kind.value,
        "origin": check.provenance.facts_origin,
        "evidence": check.evidence,
    }
    for check in learner_review.checks
    if check.result.value != "pass"
]

**Hint:** the engine may explain an absent local fact, but it must not
claim that a registry lookup or human review occurred when it did not.


## Checkpoint

You now have deterministic, versioned ground truth and a declared
correctness ceiling ready for a locked measurement—not LLM-generated
labels presented as facts. Test rows have not been opened.

**Next:** `10_capstone_model_vs_hybrid.ipynb` tests where a tiny model may
add value without owning authoritative readiness decisions.
